In [2]:
import joblib
import pandas as pd

# ============================================================
# LOAD SAVED LIGHTGBM MODEL
# ============================================================

model_file = "best_scaffold_model_LightGBM.pkl"

best_model = joblib.load(model_file)

print(best_model)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('variance', VarianceThreshold()),
                ('model',
                 LGBMRegressor(colsample_bytree=0.6, learning_rate=0.05,
                               max_depth=10, min_child_samples=50,
                               n_estimators=600, num_leaves=50,
                               objective='regression', random_state=42,
                               reg_alpha=1, reg_lambda=0, verbosity=1))])


In [3]:
# ============================================================
# EXTRACT FEATURE NAMES FROM SAVED PIPELINE
# ============================================================

# Original feature names entering the pipeline
feature_names = best_model.named_steps["imputer"].feature_names_in_

# Features retained by VarianceThreshold
variance_selector = best_model.named_steps["variance"]
support = variance_selector.get_support()

retained_features = feature_names[support]

print("Original features:", len(feature_names))
print("Features after VarianceThreshold:", len(retained_features))

Original features: 181
Features after VarianceThreshold: 181


In [4]:
# ============================================================
# EXTRACT LIGHTGBM FEATURE IMPORTANCE
# ============================================================

lgb_model = best_model.named_steps["model"]

importance = lgb_model.feature_importances_

print("Number of importance values:", len(importance))
print("Number of retained features:", len(retained_features))

Number of importance values: 181
Number of retained features: 181


In [6]:
# ============================================================
# COMPLETE FEATURE IMPORTANCE TABLE
# ============================================================

feature_importance_df = pd.DataFrame({
    "Feature": retained_features,
    "Importance": importance
})

feature_importance_df = (
    feature_importance_df
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

# Add rank
feature_importance_df.insert(
    0,
    "Rank",
    range(1, len(feature_importance_df) + 1)
)

print(feature_importance_df.head(60))

    Rank              Feature  Importance
0      1             BalabanJ         590
1      2         BCUT2D_MWLOW         575
2      3  MinAbsPartialCharge         485
3      4        BCUT2D_LOGPHI         484
4      5          BCUT2D_MRHI         473
5      6            PEOE_VSA7         473
6      7               AvgIpc         472
7      8         BCUT2D_CHGHI         470
8      9            PEOE_VSA8         464
9     10         BCUT2D_CHGLO         460
10    11       MaxEStateIndex         449
11    12         BCUT2D_MRLOW         435
12    13                  Phi         432
13    14            PEOE_VSA9         428
14    15                  qed         423
15    16              BertzCT         411
16    17          VSA_EState7         410
17    18    MinAbsEStateIndex         408
18    19       BCUT2D_LOGPLOW         405
19    20            PEOE_VSA6         404
20    21     FpDensityMorgan3         401
21    22           SlogP_VSA3         400
22    23  MaxAbsPartialCharge     

In [7]:
# ============================================================
# SAVE TOP 80 FEATURES
# ============================================================

top60_features = feature_importance_df.head(80)

top60_features.to_csv(
    "top_80_features.csv",
    index=False
)

print("\nTop 80 features saved to:")
print("top_80_features.csv")


Top 80 features saved to:
top_80_features.csv
